In [23]:
import sys
import json
import subprocess
from pathlib import Path

import pandas as pd

# Directory paths
ROOT_DIR = Path.cwd()
SRC_DIR = ROOT_DIR / "src"
DATA_DIR = ROOT_DIR / "data"


def get_json_files():
    return sorted(DATA_DIR.glob("*.json"), key=lambda p: p.stat().st_mtime, reverse=True)

In [24]:
# Run the scraper script in src/
scraper_files = sorted(SRC_DIR.glob("*.py"))
if not scraper_files:
    raise FileNotFoundError("No Python files found in 'src/'.")

for script in scraper_files:
    result = subprocess.run(
        [sys.executable, str(script)],
        capture_output=True,
        text=True
    )
    print(f"Ran: {script.name}")
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print(f"Error: {result.stderr}")

# Read the latest JSON file as pandas DataFrame
json_files = get_json_files()
if not json_files:
    raise FileNotFoundError("No JSON file found in 'data/' after running scraper.")

json_path = json_files[0]
with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

if isinstance(data, list):
    df = pd.json_normalize(data)
elif isinstance(data, dict):
    list_keys = [k for k, v in data.items() if isinstance(v, list) and (len(v) == 0 or all(isinstance(x, dict) for x in v))]
    df = pd.json_normalize(data[list_keys[0]]) if len(list_keys) == 1 else pd.json_normalize(data)
else:
    df = pd.DataFrame({"value": [data]})

print(f"Loaded: {json_path}")
df

Ran: edcs_scraper.py
[+] Connecting to EDCS API and testing page size...
[+] Page size 500 works! Total records in EDCS: 542,290
[resume] Scanning existing TSV: /Users/sanojdoddapaneni/Documents/Projects/EDCS-Analytics/data/edcs_inscriptions.tsv
[resume] Existing rows: 542,290 | Last EDCS ID: EDCS-85701225
[✓] Local data is up to date. Website rows: 542,290 | Local rows/checkpoint: 542,290

Loaded: /Users/sanojdoddapaneni/Documents/Projects/EDCS-Analytics/data/edcs_inscriptions.json


,edcs_id,monument_id,province,place,material,dating,image_count,longitude,latitude,inscription_text,...,category,subcategory,praenomen,nomen,cognomen,not_before,not_after,references,images,links
0,EDCS-00000001,1,Noricum,Maribor / Marburg,lapis,,1,15.644770,46.555628,?] C(aius) Trebonius IIvir et praef(ectus) i(u...,...,tituli sepulcrales,viri|praenomen et nomen|ordo decurionum,,,,,,ILSlov-02-01 0 00455,bilder/Mu/Muchar_1_p_398.jpg,
1,EDCS-00000002,2,NaN,NaN,lapis,,1,NaN,NaN,]ITSAMARVS[3] / [3]ETS TOVICTO[,...,tituli sacri,,,,,,,Noelke-2021 00040,bilder/No/Noelke-2021_00040.jpg,
2,EDCS-00000003,3,Germania inferior,Niederbachem / Wachtberg,lapis,,1,7.178584,50.646398,[I(ovi)] O(ptimo) M(aximo),...,tituli sacri,,,,,,,Zimmermann-2022 0 p 19,bilder/Zi/Zimmermann-2022_19.jpg,
3,EDCS-00000004,4,Germania inferior,Niederbachem / Wachtberg,opus figlinae,,0,7.178584,50.646398,L(egio) I M(inervia),...,tituli fabricationis,,,,,,,Zimmermann-2022 0 p 20,,
4,EDCS-00000005,5,Venetia et Histria / Regio X,Este / Ateste,aes,,2,11.659796,45.224007,C(aius) Artilius,...,sigilla impressa,praenomen et nomen,,,,,,ArchVen 2022 145,bilder/Ar/ArchVen-2022-145.jpg | bilder/Ar/Arc...,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
542285,EDCS-85701221,85701221,Germania superior,Eisenberg (Pfalz),lapis,,0,8.073226,49.558956,[In] h(onorem) d(omus) d(ivinae) / [M]arti Lou...,...,tituli honorarii,tituli sacri|tria nomina|viri|Augusti/Augustae,,,,,,Kreckel-2007 2004 p 421,,
542286,EDCS-85701222,85701222,Germania superior,Rheinzabern / Tabernae,lapis,,0,8.275594,49.118509,Mercur/io Toure/no Firm/o Atron[i]s,...,tituli sacri,mulieres|nomen singulare,,,,,,Brambach 01830,,
542287,EDCS-85701223,85701223,Baetica,Almensilla,lapis,"[502, 508]",1,-6.113052,37.310001,Martinus famulu/s dei vixit annos plus / minus...,...,tituli sepulcrales,inscriptiones christianae|viri|nomen singulare,,,,502,508,FE 00898,bilder/FE/FE_00898.jpg,
542288,EDCS-85701224,85701224,Hispania citerior,Banos de Rio Caldo,opus figlinae,,1,-8.107254,41.856406,[3]SCI / [3 O]gere(n)sibu[s,...,tituli possessionis,,,,,,,FE 00899,bilder/FE/FE_00899.jpg,


In [27]:
target_id = "EDCS-41200324"
df.loc[df["edcs_id"] == target_id, ["edcs_id", "inscription_text"]]

,edcs_id,inscription_text
354639,EDCS-41200324,?


In [25]:
list(df.columns)

['edcs_id',
 'monument_id',
 'province',
 'place',
 'material',
 'dating',
 'image_count',
 'longitude',
 'latitude',
 'inscription_text',
 'inscription_text_original',
 'language',
 'category',
 'subcategory',
 'praenomen',
 'nomen',
 'cognomen',
 'not_before',
 'not_after',
 'references',
 'images',
 'links']

In [26]:
# Treat both NaN and blank strings as missing values
blank_mask = df.astype("string").apply(lambda s: s.str.strip() == "")
missing_mask = df.isna() | blank_mask
missing_counts = missing_mask.sum().sort_values(ascending=False)
missing_cols = missing_counts[missing_counts > 0]

print(f"Columns with missing values (NaN or blank): {len(missing_cols)}")
missing_cols

Columns with missing values (NaN or blank): 19


cognomen                     542290
nomen                        542290
praenomen                    542290
images                       432692
not_after                    324465
not_before                   324453
dating                       324453
inscription_text_original    324443
links                        258991
subcategory                  165190
category                      96472
material                      74668
latitude                      18267
longitude                     18267
references                     4356
place                          3827
province                       3827
inscription_text                 43
language                         24
dtype: Int64

In [12]:
df.drop(columns=['inscription_text_original'], inplace=True)

In [21]:
name_cols = ["praenomen", "nomen", "cognomen"]

name_missing = (
    df[name_cols].isna()
    | (df[name_cols].astype("string").apply(lambda s: s.str.strip() == ""))
)

pd.DataFrame({
    "missing_count": name_missing.sum(),
    "total_rows": len(df),
    "missing_pct": (name_missing.sum() / len(df) * 100).round(2)
})

,missing_count,total_rows,missing_pct
praenomen,542290,542290,100.0
nomen,542290,542290,100.0
cognomen,542290,542290,100.0
